# Explore

Single-pair, fast, interactive work: load one animation, look at a crop,
time the three distance methods on one pair, and visualize what the DP
alignment actually does. Everything here should run in well under a
second to a few seconds -- the heavy, full-dataset computation lives in
`compute_pairwise.py`, run as a script, not here.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path().resolve().parent))  # project root, one level up from notebooks/

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from shapes import curves, transformations as tf, dynamic_distance as dd
from shapes.mocap_io import load_curve
from compute_pairwise import ANIMATIONS, paths_for

## Load one animation

In [ ]:
animation_id = 18  # 16_09, forward jump
asf_path, amc_path, crop = paths_for(animation_id)
c = load_curve(animation_id, asf_path, amc_path, crop)
print(ANIMATIONS[animation_id])
print("shape (joints, frames, 3, 3):", c.shape)

## Visualize the crop

Root-joint height across the recording, with the crop window shaded.
Reusable as a quick check for any animation id with a crop defined.

In [ ]:
def plot_crop(animation_id):
    info = ANIMATIONS[animation_id]
    asf_path, amc_path, crop = paths_for(animation_id)
    # load uncropped, to see the full recording including what gets trimmed
    full = load_curve(animation_id, asf_path, amc_path, crop=None)

    # root height isn't meaningful from the SO(3) curve alone (that's
    # rotation only) -- for a height-vs-frame plot like the one used to
    # derive the crop windows, read the raw .amc root translation instead.
    from shapes._dataparse import parse_amc
    anim = parse_amc(amc_path)
    root_height = np.array([f['root'][1] for f in anim.get_frames()])

    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(root_height, color="tab:blue")
    if crop:
        start, stop = crop
        ax.axvspan(start, stop or len(root_height), color="orange", alpha=0.2)
    ax.set_title(f"{info['file']} ({info['label']}), {full.shape[1]} frames")
    ax.set_xlabel("frame")
    ax.set_ylabel("root height")
    ax.grid(alpha=0.3)
    plt.show()

plot_crop(18)   # 16_09 -- the one with the extra bounce

## Time the SRVT-only distance (no alignment)

In [ ]:
import time

id_a, id_b = 18, 20  # two forward-jump trials

asf_a, amc_a, crop_a = paths_for(id_a)
asf_b, amc_b, crop_b = paths_for(id_b)
a = load_curve(id_a, asf_a, amc_a, crop_a)
b = load_curve(id_b, asf_b, amc_b, crop_b)

t0 = time.perf_counter()
d_srvt = curves.distance(a, b)
t_srvt = time.perf_counter() - t0
print(f"SRVT-only distance: {d_srvt:.4f}  ({t_srvt*1000:.1f} ms)")

## Time the SRVT+DP distance

Change `depth` and rerun to see the cost/accuracy tradeoff directly.

In [ ]:
depth = 6

t0 = time.perf_counter()
d_dp = curves.dynamic_distance(a, b, depth=depth)
t_dp = time.perf_counter() - t0
print(f"SRVT+DP distance (depth={depth}): {d_dp:.4f}  ({t_dp:.2f} s)")

## Visualize the actual DP alignment

The single most useful diagnostic in this project -- this is what caught
the "pinching" degeneracy on 16_09. A straight diagonal line means a
natural, roughly constant-speed alignment; a line that goes flat for a
while means the DP is compressing that stretch of curve `b` almost to
nothing.

In [ ]:
I0 = np.linspace(0, 1, a.shape[1])
I1 = np.linspace(0, 1, b.shape[1])
q0 = tf.skew_to_vector(tf.SRVT(a, I0))
q1 = tf.skew_to_vector(tf.SRVT(b, I1))

I1_new = dd.find_optimal_diffeomorphism(q0, q1, I0, I1, depth=depth)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(I1, I1, "--", color="gray", label="identity (no warp)")
ax.plot(I1, I1_new, color="tab:red", label="actual DP alignment")
ax.set_xlabel(f"original time in {ANIMATIONS[id_b]['file']}")
ax.set_ylabel(f"matched time in {ANIMATIONS[id_a]['file']}")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Time the signature distance, for comparison

In [ ]:
import iisignature
from shapes import log_signature as ls

k = 3  # truncation level
d = a.shape[0] * 3  # joints * 3 (vector form dimension)
s = iisignature.prepare(d, k)

t0 = time.perf_counter()
sig_a = ls.curve_log_signature(a, s)
sig_b = ls.curve_log_signature(b, s)
d_sig = ls.normalized_linear_distance(sig_a, sig_b)
t_sig = time.perf_counter() - t0
print(f"signature distance: {d_sig:.4f}  ({t_sig*1000:.1f} ms)")

## Side-by-side timing, same pair

In [ ]:
methods = ["SRVT only", f"SRVT+DP (depth={depth})", "signature"]
times = [t_srvt, t_dp, t_sig]

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(methods, times, color=["#4C72B0", "#C44E52", "#55A868"])
ax.set_xlabel("seconds (log scale)")
ax.set_xscale("log")
for y, t in enumerate(times):
    ax.text(t, y, f"  {t:.3g}s", va="center")
plt.show()